In [156]:
import numpy as np
import pandas as pd
import plotly.express as px
import os
import plotly.graph_objects as go


In [157]:
# get survex data
surveyDirectory = 'C:\\data\\udacity\\stackoverflow_survey'
surveySelection = {}
for folder in os.scandir(surveyDirectory):
    if folder.is_dir() and not str(folder.name) == "Archiv":
        surveyResults = pd.read_csv(surveyDirectory + "\\" + str(folder.name) + '\\survey_results_public.csv')
        surveySelection[str(folder.name)] = surveyResults
        

C:\Users\sce2rng\AppData\Local\Temp\ipykernel_19168\547207551.py:6: DtypeWarning:

Columns (8,12,13,14,15,16,50,51,52,53,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128) have mixed types. Specify dtype option on import or set low_memory=False.



In [158]:
#get survey question and tags
surveySchemas = {}
for folder in os.scandir(surveyDirectory):
    if folder.is_dir() and not str(folder.name) == "Archiv":
        if str(folder.name) <= "2020":
            surveySchema = pd.read_csv(surveyDirectory + "\\" + str(folder.name) + '\\survey_results_schema.csv', index_col=0)

        else:   # get only the short tag and teh question , in the newer survey schemas are also a QID and some explanations
            surveySchema = pd.read_csv(surveyDirectory + "\\" + str(folder.name) + '\\survey_results_schema.csv')
            surveySchema = surveySchema[surveySchema['qid'].str.contains('QID')]
            surveySchema = surveySchema[['qname', 'question']].set_index('qname')
            surveySchema = surveySchema.dropna()
        surveySchemas[str(folder.name)] = surveySchema.squeeze().to_dict()

In [159]:
# define functions
def getQuestionText(questionId, surveySchemas):
    """
    function to get question to the question tags
    """
    idExistPerYear = {}
    for schema in surveySchemas.keys(): # look in each year
        print(schema)
        exist = False
        if questionId in surveySchemas[schema]:
            print(surveySchemas[schema][questionId])
            exist = True
        else:
            print(f"{questionId} not found in {schema}")
        idExistPerYear[schema] = exist
    series = pd.Series(idExistPerYear, name=questionId)
    return series

In [160]:
# force numeric type to salary in 2018 and make the rest a nan
surveySelection["2018"]['Salary'] = pd.to_numeric(surveySelection["2018"]['Salary'], errors='coerce')


In [161]:
# FormalEducation is teh same as EdLevel in teh newer surveys
for year in ["2017","2018"]:
    surveySelection[year] = surveySelection[year].rename(columns={"FormalEducation":'EdLevel'})



In [162]:
# get Total anual compensation for each year, not exist for 2017
surveySelectionShort = {}
columnsShort = ["Gender", "MainBranch", "EdLevel", "FormalEducation", "Salary", "SalaryType", "YearsCode", "YearsCodePro", "Employment", "CompTotal", "CompFreq", "ConvertedComp", "LanguageWorkedWith", "LanguageDesireNextYear"]

# Filter out columns that are not present in the dataframe
for year, df in surveySelection.items():
    if year != "2017" and year != "2023":  # 2017 salary is without bonuses and benefits, 2023 has no Gender
        columnsPresent = list(df.columns)
        columnsRemove = [col for col in columnsPresent if col not in columnsShort]
        surveySelectionShort[year] = df.drop(columns=columnsRemove)

# get compensation per year for each datasaet
for year in ["2019","2020"]:
    surveySelectionShort[year] = surveySelectionShort[year].rename(columns={"ConvertedComp":'AnnualComp'})
    
surveySelectionShort["2018"] = surveySelectionShort["2018"].rename(columns={"SalaryType":'CompFreq'})
surveySelectionShort["2018"] = surveySelectionShort["2018"].rename(columns={"Salary":'CompTotal'})

def get_year_factor(freq):
    if freq == 'Weekly':
        return 52
    elif freq == 'Monthly':
        return 12
    elif freq == 'Yearly':
        return 1
    else:
        return 0
for year in ["2018","2021","2022"]:
    surveySelectionShort[year]["CompFreq"] = surveySelectionShort[year]["CompFreq"].apply(get_year_factor)
    surveySelectionShort[year]['CompFreq'] = surveySelectionShort[year].apply(lambda row: 1 if row['CompTotal'] == 0 else row['CompFreq'], axis=1)
    surveySelectionShort[year]['AnnualComp'] = surveySelectionShort[year]["CompTotal"] * surveySelectionShort[year]['CompFreq']

for year in ["2018","2019","2020","2021","2022"]:  # 2017 salary is without bonuses and benefits
    surveySelectionShort[year]['AnnualComp'] = surveySelectionShort[year]['AnnualComp'].astype('float64')
    surveySelectionShort[year]['AnnualComp'].replace(0, np.nan, inplace=True)

In [163]:
for key in surveySelectionShort.keys():
    print(key)
    print(surveySelectionShort[key].columns)

2018
Index(['Employment', 'EdLevel', 'CompTotal', 'CompFreq', 'LanguageWorkedWith',
       'LanguageDesireNextYear', 'Gender', 'AnnualComp'],
      dtype='object')
2019
Index(['MainBranch', 'Employment', 'EdLevel', 'YearsCode', 'YearsCodePro',
       'CompTotal', 'CompFreq', 'AnnualComp', 'LanguageWorkedWith',
       'LanguageDesireNextYear', 'Gender'],
      dtype='object')
2020
Index(['MainBranch', 'CompFreq', 'CompTotal', 'AnnualComp', 'EdLevel',
       'Employment', 'Gender', 'LanguageDesireNextYear', 'LanguageWorkedWith',
       'YearsCode', 'YearsCodePro'],
      dtype='object')
2021
Index(['MainBranch', 'Employment', 'EdLevel', 'YearsCode', 'YearsCodePro',
       'CompTotal', 'CompFreq', 'Gender', 'AnnualComp'],
      dtype='object')
2022
Index(['MainBranch', 'Employment', 'EdLevel', 'YearsCode', 'YearsCodePro',
       'CompTotal', 'CompFreq', 'Gender', 'AnnualComp'],
      dtype='object')


In [164]:
print(surveySelectionShort["2021"]["Gender"].value_counts())
nanCount = surveySelectionShort["2021"]["Gender"].isna().sum()
print(f"nans: {nanCount}")

Man                                                                                   74817
Woman                                                                                  4120
Prefer not to say                                                                      1442
Non-binary, genderqueer, or gender non-conforming                                       690
Or, in your own words:                                                                  413
Man;Or, in your own words:                                                              268
Man;Non-binary, genderqueer, or gender non-conforming                                   252
Woman;Non-binary, genderqueer, or gender non-conforming                                 147
Man;Woman                                                                                41
Non-binary, genderqueer, or gender non-conforming;Or, in your own words:                 21
Man;Woman;Non-binary, genderqueer, or gender non-conforming                     

In [165]:
import copy

surveySelectionShort = copy.deepcopy(surveySelectionShort)

def align_gender(gender):
    if gender == "Man":
        return "Male"
    elif gender == "Woman":
        return "Female"
    elif gender == "Male":
        return "Male"
    elif gender == "Female":
        return "Female"
    elif gender == "Prefer not to say":
        return None
    elif pd.isnull(gender):
        return None
    else:
        return "Other"
    
for year in surveySelectionShort.keys():
    surveySelectionShort[year]["GenderCat"] = surveySelectionShort[year]["Gender"].apply(align_gender)

print(surveySelectionShort["2021"]["GenderCat"].value_counts())
nanCount = surveySelectionShort["2021"]["e"].isna().sum()
print(f"nans: {nanCount}")


Male      74817
Female     4120
Other      1907
Name: GenderCat, dtype: int64


KeyError: 'e'

In [ ]:
getQuestionText("Employment", surveySchemas)

2017
Employment not found in 2017
2018
Which of the following best describes your current employment status?
2019
Which of the following best describes your current employment status?
2020
Which of the following best describes your current employment status?
2021
Which of the following best describes your current <b>employment status</b>?
2022
Which of the following best describes your current employment status?
2023
Which of the following best describes your current employment status? Select all that apply.


2017    False
2018     True
2019     True
2020     True
2021     True
2022     True
2023     True
Name: Employment, dtype: bool

In [ ]:
surveySelectionShort["2022"]["Employment"].value_counts()

Employed, full-time                                                                                                                    42962
Student, full-time                                                                                                                      6756
Independent contractor, freelancer, or self-employed                                                                                    4978
Employed, full-time;Independent contractor, freelancer, or self-employed                                                                3486
Not employed, but looking for work                                                                                                      1831
                                                                                                                                       ...  
Student, part-time;Independent contractor, freelancer, or self-employed;Retired                                                            1
Employed, ful

In [ ]:
# handel not employed
def align_employment(employment):
    if employment == "Employed full-time" or employment == "Employed part-time" or employment == "Independent contractor, freelancer, or self-employed":
        return "Employed"
    elif employment == "Employed, full-time" or employment == "Employed, part-time":
        return "Employed"
    elif employment == "Not employed, and not looking for work" or employment == "Not employed, but looking for work":
        return "Unemployed"
    elif employment == "Retired":
        return "Retired"
    elif employment == "Student" or employment == "Student, full-time" or employment == "Student, part-time":
        return "Student"
    elif employment == "I prefer not to say":
        return None
    elif pd.isnull(employment):
        return None
    elif employment.count("full-time") > 1 or ("Not employed," in employment and "Employed," in employment):
        return None
    elif "Employed," in employment:
        return "Employed"
    else:
        return None

for year in surveySelectionShort.keys():
    surveySelectionShort[year]["EmpUnemp"] = surveySelectionShort[year]["Employment"].apply(align_employment)
surveySelectionShort[year]["EmpUnemp"].value_counts()

Employed      56573
Student        7801
Unemployed     2548
Retired         305
Name: EmpUnemp, dtype: int64

In [ ]:
# Get median values for annual compensation per year and gender
annualComp = {}
for year, df in surveySelectionShort.items():
    print(len(df))
    annualComp[year] = df[df['EmpUnemp'] == "Employed"]
    print(len(annualComp[year]))
    annualComp[year] = annualComp[year].groupby('GenderCat')["AnnualComp"].median()

98855
85157
88883
77420
64461
53159
83439
64086
73268
56573


In [ ]:
# make one dataframe with with year, gender and annual compensation
data = []
for year, compSeries in annualComp.items():
    # get annual compensation for each year and gender-group
    for gender, comp in compSeries.items():
        data.append({'Year': year, 'Gender': gender, 'AnnualComp': comp})

df = pd.DataFrame(data)

# creat plot with annual compensation over year with gender in different colors
fig = go.Figure()
fig = px.scatter(df, x='Year', y='AnnualComp', color='Gender',
                 title='Annual Compensation by Gender Over Years',
                 labels={'AnnualComp': 'Annual Compensation Median', 'Gender': 'Gender'})

# Zeigen Sie den Plot an
fig.show()